In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/data"
SRC_DIR = "/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/src"
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer"

In [ ]:
!pip install -q torchmetrics transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.3 MB/s eta 0:00:00


In [ ]:
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import AutoTokenizer, AutoModel
from torchvision import transforms

from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm import tqdm

In [ ]:
import sys
sys.path.append('/content')
from dataset import AdImageDataset
from image_model import AdImageModel

ModuleNotFoundError: No module named 'dataset'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [ ]:
# tiny dataset (exactly 5 images)
full_ds = AdImageDataset(csv_path=DATA_DIR + "/splits/train.csv", img_dir=DATA_DIR + "/raw")
tiny_ds = Subset(full_ds, range(5))  # Grabs the first 5 rows from train.csv
tiny_loader = DataLoader(tiny_ds, batch_size=5, shuffle=False, num_workers=0)

In [ ]:
model = AdImageModel(num_content_types=3, num_moods=4, freeze_encoder=True).to(device)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 87.3MB/s]


In [ ]:
criterion_content = nn.CrossEntropyLoss()
criterion_mood = nn.CrossEntropyLoss()

In [ ]:
NUM_CONTENT = 3
NUM_MOODS = 4

In [ ]:
def get_device(requested: str = "auto") -> str:
    if requested == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return requested

In [ ]:
def calculate_class_weights(
    csv_path, column_name, num_classes, device: str
) -> torch.Tensor:
    """Compute inverse-frequency weights from training CSV."""
    df = pd.read_csv(csv_path)

    # Fill missing classes with 0 to avoid index errors
    counts = df[column_name].value_counts().reindex(range(num_classes), fill_value=0)

    counts = torch.tensor(counts.values, dtype=torch.float32)
    counts = counts.clamp(min=1)

    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes  # normalize
    return weights.to(device)

In [ ]:
def train_epoch(model, loader, optimizer, criterion_content, criterion_mood, device):
    """Train for one epoch and return average loss."""
    model.train()
    total_loss = 0.0
    correct_content = 0
    correct_mood = 0
    total_samples = 0

    for batch in loader:
        image = batch["image"].to(device)
        mood_labels = batch["mood"].to(device)
        content_labels = batch["content_type"].to(device)

        optimizer.zero_grad()

        content_logits, mood_logits = model(image)

        loss_content = criterion_content(content_logits, content_labels)
        loss_mood = criterion_mood(mood_logits, mood_labels)
        loss = loss_content + loss_mood

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Metrics
        _, content_preds = torch.max(content_logits, 1)
        _, mood_preds = torch.max(mood_logits, 1)

        correct_content += (content_preds == content_labels).sum().item()
        correct_mood += (mood_preds == mood_labels).sum().item()
        total_samples += image.size(0)

    avg_loss = total_loss / len(loader)
    content_acc = 100.0 * correct_content / total_samples
    mood_acc = 100.0 * correct_mood / total_samples
    return avg_loss, content_acc, mood_acc


In [ ]:
try:
    avg_loss, content_acc, mood_acc = train_epoch(
        model, tiny_loader, optimizer, criterion_content, criterion_mood, device
    )
    print("\n✅ TEST PASSED!")
    print(f"📉 Avg Loss: {avg_loss:.4f}")
    print(f"🎯 Content Acc: {content_acc:.1f}%")
    print(f"🎨 Mood Acc: {mood_acc:.1f}%")
except Exception as e:
    print(f"\n❌ TEST FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


✅ TEST PASSED!
📉 Avg Loss: 2.6423
🎯 Content Acc: 0.0%
🎨 Mood Acc: 60.0%


In [ ]:
@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()

    f1_ct = MulticlassF1Score(num_classes=NUM_CONTENT, average="weighted").to(device)
    f1_mood = MulticlassF1Score(num_classes=NUM_MOODS, average="weighted").to(device)

    for batch in loader:
        imgs = batch["image"].to(device)
        ct_labels = batch["content_type"].to(device)
        m_labels = batch["mood"].to(device)

        content_logits, mood_logits = model(imgs)

        f1_ct.update(content_logits, ct_labels)
        f1_mood.update(mood_logits, m_labels)

    return f1_ct.compute().item(), f1_mood.compute().item()

In [ ]:
!pip install torchmetrics
from torchmetrics.classification import MulticlassF1Score

val_ds = AdImageDataset(csv_path=DATA_DIR + "/splits/val.csv", img_dir=DATA_DIR + "/raw")
val_loader = DataLoader(val_ds, batch_size=3, shuffle=False, num_workers=0)

val_f1_ct, val_f1_mood = eval_epoch(model, val_loader, device)
print(f"Val F1 - Content: {val_f1_ct:.4f} | Mood: {val_f1_mood:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 8.6 MB/s eta 0:00:00
Val F1 - Content: 0.3439 | Mood: 0.1930


In [ ]:
# Load model
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).to(device)
model.classifier = torch.nn.Identity()
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
# Load alignment pairs CSV
# Expected columns: image_name, caption
ALIGNMENT_CSV = f'{DRIVE_DIR}/data/alignment_pairs/alignment_pairs.csv'

In [ ]:
df = pd.read_csv(ALIGNMENT_CSV)
print(f"Loaded {len(df)} alignment pairs")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

image_vectors = []
valid_idx = []

print("Extracting vectors for manual pairs")

Loaded 201 alignment pairs
Extracting vectors for manual pairs


In [ ]:
df = df[:5]
df

,image_name,caption
0,447769129_1450818865822718_174348548869986575_...,NEW COLLECTION WHOLESALE /TOPTAN \ 2024 KOLE...
1,638126494_919236147314873_1111902097198219191_...,JUST LAUNCHED: The Lion Has Arrived.\n\nIntrod...
2,665919730_1396078782534479_8541386423595437487...,Some men wear luxury. Others wear Superglamour...
3,617162607_1556299925698570_8555959236959649361...,"Inspired by Milan, a city where heritage meets..."
4,553995084_1324231542614743_6955118679336700576...,Our new collection is out. This is not a palet...


In [ ]:
with torch.no_grad():
  for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
      img_path = f'{DRIVE_DIR}/data/raw/{row["image_name"]}'
      print(img_path)
      try:
          img = Image.open(img_path).convert('RGB')
          img_t = transform(img).unsqueeze(0).to(device)
          vec = model(img_t).squeeze(0).cpu()
          image_vectors.append(vec)
          valid_idx.append(idx)
      except Exception as e:
          print(f"  Skip {row['img_path']}: {e}")

  image_vectors = torch.stack(image_vectors)  # [N, 1280]
  df_valid = df.iloc[valid_idx].reset_index(drop=True)

  # Save
  torch.save(image_vectors, f'{DRIVE_DIR}/checkpoints/alignment_image_vectors.pt')
  df_valid.to_csv(f'{DRIVE_DIR}/data/alignment_pairs/alignment_pairs_valid.csv', index=False)

  print(f"\n {len(image_vectors)} image vectors saved → Drive")
  print(f"   Shape: {image_vectors.shape}")
  print()

Extracting:  20%|██        | 1/5 [00:00<00:00,  9.82it/s]

/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/data/raw/447769129_1450818865822718_174348548869986575_n.jpg
/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/data/raw/638126494_919236147314873_1111902097198219191_n.jpg


Extracting:  80%|████████  | 4/5 [00:00<00:00,  9.61it/s]

/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/data/raw/665919730_1396078782534479_8541386423595437487_n.jpg
/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/data/raw/617162607_1556299925698570_8555959236959649361_n.jpg
/content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/data/raw/553995084_1324231542614743_6955118679336700576_n.jpg


Extracting: 100%|██████████| 5/5 [00:00<00:00,  8.61it/s]



 5 image vectors saved → Drive
   Shape: torch.Size([5, 1280])



In [ ]:
df_valid.head()

,image_name,caption
0,447769129_1450818865822718_174348548869986575_...,NEW COLLECTION WHOLESALE /TOPTAN \ 2024 KOLE...
1,638126494_919236147314873_1111902097198219191_...,JUST LAUNCHED: The Lion Has Arrived.\n\nIntrod...
2,665919730_1396078782534479_8541386423595437487...,Some men wear luxury. Others wear Superglamour...
3,617162607_1556299925698570_8555959236959649361...,"Inspired by Milan, a city where heritage meets..."
4,553995084_1324231542614743_6955118679336700576...,Our new collection is out. This is not a palet...
